# Pipeline Landing to Bronze - Ingestão de Dados Brutos

## Objetivo
Este notebook realiza a ingestão de dados da camada **Landing** (arquivos CSV brutos) para a camada **Bronze** (tabelas Delta Lake), incluindo:
- Leitura de 9 arquivos CSV do dataset Olist
- Ingestão de cotação do dólar via API do Banco Central do Brasil
- Adição de metadados de rastreabilidade (`timestamp_ingestion`)

## Arquitetura Medallion
- **Landing**: Arquivos brutos em Volumes sem transformação
- **Bronze**: Tabelas Delta Lake com schema inferido + timestamp de ingestão
- **Objetivo**: Preservar dados originais com rastreabilidade para auditoria

---

# Importação de Bibliotecas

## Bibliotecas Utilizadas
- **requests**: Comunicação HTTP com a API do Banco Central
- **pyspark.sql.functions**: Transformações de DataFrames (timestamps, renomeação)
- **pyspark.sql.types**: Definição de schemas (StructType, DoubleType, etc.)
- **datetime**: Manipulação de datas para consulta da API

In [0]:
import requests
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType
from datetime import datetime, timedelta


# Configuração de Paths e Variáveis

## Variáveis de Ambiente

| Variável | Valor | Descrição |
|----------|-------|-------------|
| `Catalogo_Name` | `visagio` | Nome do catálogo Unity Catalog onde as tabelas serão criadas |
| `DB_Bronze` | `bronze` | Nome do schema/database da camada Bronze |
| `Landing_Path` | `/Volumes/visagio/landing/atividade_rocketlab/` | Caminho do Volume Unity Catalog onde estão os arquivos CSV brutos |

## Estrutura de Dados
```
visagio (Catalog)
└── bronze (Schema)
    ├── tb_customers
    ├── tb_geolocalizacao
    ├── tb_order_items
    ├── tb_order_payments
    ├── tb_order_reviews
    ├── tb_orders
    ├── tb_products
    ├── tb_sellers
    ├── tb_product_category_name_translation
    └── tb_cotacao_dolar
```

## Regra de Negócio
- **Nomenclatura**: Todas as tabelas Bronze recebem prefixo `tb_` para identificação visual
- **Unity Catalog**: Uso obrigatório de catalogo.schema.tabela para governança de dados

In [0]:
Catalogo_Name = "visagio"
DB_Bronze = "bronze"
Landing_Path = "/Volumes/visagio/landing/atividade_rocketlab/"

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{Catalogo_Name}`.`{DB_Bronze}`")
print(f"Schema {Catalogo_Name}.{DB_Bronze} criado/verificado com sucesso!")

Schema visagio.bronze criado/verificado com sucesso!


# Ingestão De Dados

Função responsável pela leitura de arquivos CSV da camada Landing e gravação na camada Bronze.

Cada tabela recebe a coluna `data_ingestao` com o timestamp exato do momento da leitura, garantindo rastreabilidade da origem do dado.

## Parâmetros
| Parâmetro | Descrição |
|---|---|
| `nome_arquivo` | Nome exato do arquivo CSV na camada Landing |
| `nome_tabela` | Nome da tabela Delta a ser criada na camada Bronze |

## Observações
- Arquivos vazios são detectados e interrompem o processamento com mensagem de erro
- O schema é sobrescrito automaticamente (`overwriteSchema`) para tolerância a mudanças
- Não há retorno — o resultado é indicado via print de sucesso `✅` ou erro `❌`

In [0]:
def ingest_landing_to_bronze(nome_arquivo, nome_tabela):
    try:
        table_name = nome_tabela
        landing_path = f"{Landing_Path}{nome_arquivo}"

        df = spark.read.csv(
            landing_path,
            header=True,
            inferSchema=True,
            quote='"',
            escape='"',
            multiLine=True
        )

        if df.count() == 0:
            raise ValueError(f"Arquivo Vazio, verifique o arquivo {landing_path}")

        df_with_metadata = df.withColumn("timestamp_ingestion", F.current_timestamp())

        df_with_metadata.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{Catalogo_Name}.{DB_Bronze}.{table_name}")

        print(f"✅ Tabela bronze.{nome_tabela} criada com sucesso!\n")

    except Exception as e:
        print(f"Erro ao processar {nome_tabela}: {str(e)}")

# Execução da Ingestão - Dataset Olist E-Commerce

## Tabelas a Serem Ingeridas

Este bloco executa a ingestão de **9 tabelas** do dataset público Olist Brazilian E-Commerce:

### 1. Dimensões (Dados Mestres)
| Tabela Bronze | Arquivo CSV | Descrição |
|---------------|-------------|-------------|
| `tb_customers` | `olist_customers_dataset.csv` | Dados cadastrais dos clientes (nome, localização, gênero) |
| `tb_products` | `olist_products_dataset.csv` | Catálogo de produtos (categoria, dimensões, peso) |
| `tb_sellers` | `olist_sellers_dataset.csv` | Cadastro de vendedores/sellers (localização) |
| `tb_geolocalizacao` | `olist_geolocation_dataset.csv` | Coordenadas geográficas por CEP |
| `tb_product_category_name_translation` | `product_category_name_translation.csv` | Tradução de categorias PT → EN |

### 2. Fatos (Transações)
| Tabela Bronze | Arquivo CSV | Descrição |
|---------------|-------------|-------------|
| `tb_orders` | `olist_orders_dataset.csv` | Pedidos com datas de compra, aprovação e entrega |
| `tb_order_items` | `olist_order_items_dataset.csv` | Itens dos pedidos (produto, preço, frete) |
| `tb_order_payments` | `olist_order_payments_dataset.csv` | Pagamentos dos pedidos (tipo, parcelas, valor) |
| `tb_order_reviews` | `olist_order_reviews_dataset.csv` | Avaliações dos clientes (nota, comentário) |

## Regras de Negócio Aplicadas

### 1. Leitura de CSV
- **header=True**: Primeira linha contém nomes de colunas
- **inferSchema=True**: Spark detecta automaticamente tipos de dados (int, string, timestamp)
- **multiLine=True**: Suporte a campos CSV com quebras de linha (comentários de reviews)
- **quote='"' e escape='"'**: Tratamento de aspas duplas em campos de texto

### 2. Metadados de Auditoria
- **timestamp_ingestion**: Carimbo de tempo exato da leitura do arquivo
- **Justificativa**: Rastreabilidade para debugging e auditoria ("quando este dado entrou?")

### 3. Gravação Delta Lake
- **Formato**: Delta Lake (ACID transactions, time travel, schema evolution)
- **Modo**: Overwrite (recria tabela do zero a cada execução)
- **overwriteSchema=true**: Tolera mudanças de schema entre execuções

## Validações Implementadas
- **Arquivo vazio**: Lança `ValueError` se `df.count() == 0`
- **Tratamento de erros**: Try/except com mensagem clara de falha

## Execução Sequencial
As 9 tabelas são processadas **sequencialmente** (não em paralelo) para facilitar debugging e logs ordenados.

In [0]:
ingest_landing_to_bronze("olist_customers_dataset.csv",           "tb_customers")
ingest_landing_to_bronze("olist_geolocation_dataset.csv",          "tb_geolocalizacao")
ingest_landing_to_bronze("olist_order_items_dataset.csv",          "tb_order_items")
ingest_landing_to_bronze("olist_order_payments_dataset.csv",       "tb_order_payments")
ingest_landing_to_bronze("olist_order_reviews_dataset.csv",        "tb_order_reviews")
ingest_landing_to_bronze("olist_orders_dataset.csv",               "tb_orders")
ingest_landing_to_bronze("olist_products_dataset.csv",             "tb_products")
ingest_landing_to_bronze("olist_sellers_dataset.csv",              "tb_sellers")
ingest_landing_to_bronze("product_category_name_translation.csv",  "tb_product_category_name_translation")

✅ Tabela bronze.tb_customers criada com sucesso!

✅ Tabela bronze.tb_geolocalizacao criada com sucesso!

✅ Tabela bronze.tb_order_items criada com sucesso!

✅ Tabela bronze.tb_order_payments criada com sucesso!

✅ Tabela bronze.tb_order_reviews criada com sucesso!

✅ Tabela bronze.tb_orders criada com sucesso!

✅ Tabela bronze.tb_products criada com sucesso!

✅ Tabela bronze.tb_sellers criada com sucesso!

✅ Tabela bronze.tb_product_category_name_translation criada com sucesso!



# Ingestão da Cotação do Dólar (API Banco Central do Brasil)

## Objetivo
Consultar a **API PTAX do Banco Central** para obter cotações diárias do dólar (USD → BRL) no período dos pedidos.

## Por Que Cotação do Dólar?
- **Regra de Negócio**: Dataset Olist contém preços em BRL, mas análises internacionais requerem conversão para USD
- **Fonte oficial**: API do Banco Central é a fonte mais confiável para cotações históricas no Brasil
- **Granularidade**: Cotações diárias com horário de fechamento
## Estratégia de Ingestão Dinâmica

Antes de consultar a API, o notebook **identifica automaticamente o intervalo de datas** do dataset:

### Etapa 1: Identificar Range de Datas
1. Consulta a tabela `bronze.tb_orders` (já ingerida)
2. Calcula `MIN(order_purchase_timestamp)` e `MAX(order_purchase_timestamp)`
3. Resultado: Define o período exato dos pedidos no dataset

### Etapa 2: Consultar API do BCB
1. Usa o range identificado para montar a URL da API PTAX
2. **Extrapolação**: Adiciona margem na `data_fim` para cobrir possíveis registros futuros
3. **Motivo**: Garantir que todas as datas do dataset tenham cotação disponível

## Transformações Aplicadas

| Campo Original | Campo Bronze | Transformação | Justificativa |
|----------------|--------------|----------------|---------------|
| `dataHoraCotacao` | `data_hora_cotacao` | `to_timestamp()` | Padronizar formato timestamp |
| - | `data_cotacao` | `date_format("MM-dd-yyyy")` | Data sem hora para joins |
| `cotacaoCompra` | `cotacao_compra` | `cast(DoubleType())` | Forçar tipo numérico |
| - | `data_ingestao` | `current_timestamp()` | Metadado de auditoria |

## Validações Implementadas
1. **Status HTTP**: Lança `ConnectionError` se API retornar erro (status ≠ 200)
2. **Dados vazios**: Lança `ValueError` se JSON não contém campo `value`
3. **DataFrame vazio**: Verifica `df.count() > 0` após transformações
4. **Timeout**: Limite de 30 segundos para requisição HTTP

In [0]:
def ingest_cotacao_dolar(data_inicio_formatada, data_fim_formatada):
    try:
        url = (
            "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
            f"CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)"
            f"?@dataInicial='{data_inicio_formatada}'"
            f"&@dataFinalCotacao='{data_fim_formatada}'"
            f"&$select=dataHoraCotacao,cotacaoCompra"
            f"&$format=json"
        )

        response = requests.get(url, timeout=30)

        if response.status_code != 200:
            raise ConnectionError(f"Erro na API do BCB: HTTP {response.status_code}")

        dados = response.json().get("value", [])

        if not dados:
            raise ValueError(f"Nenhuma cotação retornada para o período {data_inicio_formatada} a {data_fim_formatada}")

        df = spark.createDataFrame(dados)

        df = (
            df
            .withColumnRenamed("dataHoraCotacao", "data_hora_cotacao")
            .withColumnRenamed("cotacaoCompra",   "cotacao_compra")
            .withColumn("data_hora_cotacao", F.to_timestamp("data_hora_cotacao"))
            .withColumn("data_cotacao",      F.date_format(F.to_date("data_hora_cotacao"), "MM-dd-yyyy"))
            .withColumn("cotacao_compra",    F.col("cotacao_compra").cast(DoubleType()))
            .withColumn("data_ingestao",     F.current_timestamp())
        )

        if df.count() == 0:
            raise ValueError("DataFrame vazio após transformações")

        df.write.format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(f"{Catalogo_Name}.{DB_Bronze}.tb_cotacao_dolar")

        print(f"✅ Tabela bronze.tb_cotacao_dolar criada com sucesso! ({df.count()} registros)\n")

    except Exception as e:
        print(f"❌ Erro ao ingerir cotação do dólar: {str(e)}")

# Configuração de Parâmetros (Widgets Databricks)

## Objetivo
Criar **widgets de entrada** para parametrizar as datas da consulta à API do BCB, permitindo ajustes sem modificar o código.

## Widgets Criados

| Nome | Valor Padrão | Formato | Descrição |
|------|---------------|---------|-------------|
| `data_inicio` | 09-01-2016 | MM-DD-AAAA | Data inicial para consulta da API PTAX |
| `data_fim` | 12-31-2018 | MM-DD-AAAA | Data final para consulta da API PTAX |

## Regras de Negócio

### 1. Valores Padrão
- **data_inicio**: Ajustada para cobrir o menor `order_purchase_timestamp` do dataset
- **data_fim**: Ajustada para cobrir o maior `order_purchase_timestamp` + margem de segurança
- **Justificativa**: Garantir que **todas as datas dos pedidos** tenham cotação disponível

### 2. Formato de Data
- **Padrão API BCB**: MM-DD-AAAA (formato americano)
- **Exemplo**: 09-01-2016 = 1º de setembro de 2016
- **Atenção**: Diferente do formato brasileiro (DD-MM-AAAA)

## Validação de Range

O próximo bloco de código realiza **validação dinâmica** do range:

### Etapas
1. **Consulta Bronze**: Lê `bronze.tb_orders` para obter datas reais do dataset
2. **Cálculo de MIN/MAX**: Identifica o intervalo exato dos pedidos
3. **Comparação**: Exibe lado a lado:
   - Range real do dataset (datas dos pedidos)
   - Range configurado nos widgets (datas da API)
4. **Verificação manual**: Permite ao usuário confirmar se os widgets estão corretos

## Beneficios dos Widgets
- ✅ **Flexibilidade**: Ajustar datas sem reexecutar células anteriores
- ✅ **Documentação visual**: Interface gráfica mostra parâmetros ativos
- ✅ **Jobs parametrizados**: Widgets permitem criar Jobs Databricks com parâmetros dinâmicos
- ✅ **Debugging**: Fácil testar diferentes ranges sem alterar código

In [0]:
dbutils.widgets.text("data_inicio", "09-01-2016", "Data Início (MM-DD-AAAA)")
dbutils.widgets.text("data_fim",    "12-31-2018", "Data Fim (MM-DD-AAAA)")

In [0]:
df_orders = spark.table(f"{Catalogo_Name}.{DB_Bronze}.tb_orders")

resultado = df_orders.agg(
    F.min("order_purchase_timestamp").alias("data_min"),
    F.max("order_purchase_timestamp").alias("data_max")
).collect()[0]

print(f"📅 Range real do dataset:")
print(f"   Mínima : {resultado['data_min']}")
print(f"   Máxima : {resultado['data_max']}")
print(f"\n🔧 Parâmetros configurados nos widgets:")
print(f"   data_inicio : {dbutils.widgets.get('data_inicio')}")
print(f"   data_fim    : {dbutils.widgets.get('data_fim')}")

📅 Range real do dataset:
   Mínima : 2016-09-04 21:15:19
   Máxima : 2018-10-17 17:30:18

🔧 Parâmetros configurados nos widgets:
   data_inicio : 09-04-2016
   data_fim    : 12-31-2018


# Execução da Ingestão - Cotação do Dólar

## Objetivo
Executar a consulta à API do Banco Central usando os parâmetros definidos nos widgets e gravar os dados na tabela Bronze.

## Processo de Execução

### 1. Leitura dos Widgets
```python
data_inicio_formatada = dbutils.widgets.get("data_inicio")  # Ex: "09-01-2016"
data_fim_formatada    = dbutils.widgets.get("data_fim")     # Ex: "12-31-2018"
```

### 2. Construção da URL da API
A função `ingest_cotacao_dolar()` monta a URL dinamicamente:
```
https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/
CotacaoDolarPeriodo(dataInicial='09-01-2016',dataFinalCotacao='12-31-2018')
?$select=dataHoraCotacao,cotacaoCompra&$format=json
```

### 3. Requisição HTTP
- **Método**: GET
- **Timeout**: 30 segundos
- **Formato resposta**: JSON
- **Status esperado**: 200 (OK)

### 4. Processamento da Resposta
1. **Parse JSON**: Extrai campo `value` (lista de cotações)
2. **Cria DataFrame**: `spark.createDataFrame(dados)`
3. **Transformações**:
   - Renomear colunas para padrão Bronze
   - Converter tipos de dados (timestamp, double)
   - Adicionar campo `data_ingestao`
4. **Gravação**: `saveAsTable("visagio.bronze.tb_cotacao_dolar")`

## Extrapolação da Data Final

### Estratégia Adotada
- **data_inicio**: Usa a data mínima do dataset (sem ajuste)
- **data_fim**: **Extrapola** até 31/12/2018 (mesmo que MAX seja anterior)

### Justificativa da Extrapolação
1. **Segurança**: Cobre possíveis registros tardios ou atualizações do dataset
2. **Consistência**: Evita erros de join na camada Silver (falta de cotação para alguma data)
3. **Custo baixo**: Alguns dias extras de cotação não impactam performance
4. **Previsão**: Facilita análises que extrapolam o período exato do dataset

## Exemplo de Dados Retornados

```json
{
  "value": [
    {
      "dataHoraCotacao": "2016-09-01 13:09:37.192",
      "cotacaoCompra": 3.2387
    },
    {
      "dataHoraCotacao": "2016-09-02 13:05:12.441",
      "cotacaoCompra": 3.2612
    }
  ]
}
```

## Resultado Final
Tabela `visagio.bronze.tb_cotacao_dolar` com:
- **Colunas**: `data_hora_cotacao`, `data_cotacao`, `cotacao_compra`, `data_ingestao`
- **Período**: 09/2016 a 12/2018 (aprox. 850 registros diários)
- **Formato**: Delta Lake
- **Uso futuro**: Join com `bronze.tb_orders` na camada Silver para converter BRL → USD

In [0]:
data_inicio_formatada = dbutils.widgets.get("data_inicio")
data_fim_formatada    = dbutils.widgets.get("data_fim")

print(f"🔗 Período para API do BCB:")
print(f"   Início : {data_inicio_formatada}")
print(f"   Fim    : {data_fim_formatada}")

ingest_cotacao_dolar(data_inicio_formatada, data_fim_formatada)

🔗 Período para API do BCB:
   Início : 09-04-2016
   Fim    : 12-31-2018
✅ Tabela bronze.tb_cotacao_dolar criada com sucesso! (580 registros)

